# 🔥 Notebook 03 — Fine-tuning Completo

**Objetivo:** Superar el baseline descongelando gradualmente el backbone.

**Estrategia:**
1. Cargar el checkpoint `baseline.pt`
2. Descongelar los últimos 3 bloques de EfficientNet-B0
3. Usar **learning rates diferenciales**: backbone lr bajo (1e-4), cabeza lr alto (1e-3)
4. Scheduler `CosineAnnealingLR` para decaer el lr suavemente
5. Early stopping con paciencia de 5 épocas

**Meta:** Top-1 > 80%, Top-5 > 92%


In [ ]:
import sys
sys.path.append('..')

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision import datasets
from torch.utils.data import DataLoader

from src.config import (
    DATA_DIR, WEIGHTS_DIR, DEVICE, SEED, BATCH_SIZE, FOOD101_CLASSES,
    FINETUNE_BACKBONE_LR, FINETUNE_HEAD_LR, FINETUNE_WEIGHT_DECAY,
    FINETUNE_EPOCHS, EARLY_STOP_PATIENCE, LABEL_SMOOTHING
)
from src.model import load_model, unfreeze_last_blocks, get_param_groups
from src.trainer import train_model
from src.transforms import get_train_transform, get_val_transform
from src.utils import plot_training_curves

torch.manual_seed(SEED)
print(f'Device: {DEVICE}')

## 📁 Parte 1 — DataLoaders


In [ ]:
train_loader = DataLoader(
    datasets.Food101(str(DATA_DIR), split='train', transform=get_train_transform(), download=False),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    datasets.Food101(str(DATA_DIR), split='test', transform=get_val_transform(), download=False),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True
)
print(f'Train: {len(train_loader.dataset):,} | Val: {len(val_loader.dataset):,}')

## 🧠 Parte 2 — Cargar Baseline y Descongelar Backbone


In [ ]:
baseline_path = WEIGHTS_DIR / 'baseline.pt'
assert baseline_path.exists(), f'Primero correr 02_baseline.ipynb — no se encontró {baseline_path}'

model = load_model(baseline_path, backbone='efficientnet_b0', device=DEVICE)

# Descongelar los últimos 3 bloques de features + classifier
# EfficientNet-B0: features[0..8] → dejamos libres features[6], [7], [8]
unfreeze_last_blocks(model, n_blocks=3, backbone='efficientnet_b0')

total      = sum(p.numel() for p in model.parameters())
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros entrenables: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)')

## 🏋️ Parte 3 — Optimizador con Learning Rates Diferenciales

La idea es que las capas del backbone ya tienen representaciones útiles:
un lr muy alto las destruiría. La cabeza, en cambio, necesita aprender desde cero.


In [ ]:
param_groups = get_param_groups(
    model,
    backbone_lr=FINETUNE_BACKBONE_LR,  # 1e-4
    head_lr=FINETUNE_HEAD_LR,          # 1e-3
    backbone='efficientnet_b0'
)

optimizer = torch.optim.AdamW(param_groups, weight_decay=FINETUNE_WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINETUNE_EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

print(f'Optimizer: AdamW  backbone_lr={FINETUNE_BACKBONE_LR}  head_lr={FINETUNE_HEAD_LR}')
print(f'Scheduler: CosineAnnealingLR  T_max={FINETUNE_EPOCHS}')
print(f'Early stopping: paciencia={EARLY_STOP_PATIENCE} épocas')

## 🚀 Parte 4 — Entrenamiento


In [ ]:
save_path = WEIGHTS_DIR / 'model_v1.pt'

history = train_model(
    model, train_loader, val_loader,
    epochs=FINETUNE_EPOCHS,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    device=DEVICE,
    save_path=save_path,
    patience=EARLY_STOP_PATIENCE,
)

## 📈 Parte 5 — Curvas y Comparativa con Baseline


In [ ]:
fig = plot_training_curves(history)
plt.show()

best_top1 = max(history['val_top1'])
best_top5 = max(history['val_top5'])
best_epoch = history['val_top1'].index(best_top1) + 1

print(f'\n=== Mejor época: {best_epoch} ===')
print(f'Val Top-1: {best_top1:.4f} ({best_top1*100:.2f}%)')
print(f'Val Top-5: {best_top5:.4f} ({best_top5*100:.2f}%)')
print(f'\nMeta Top-1 > 80%: {"✓" if best_top1 > 0.80 else "✗ (seguir entrenando)"}')
print(f'Modelo guardado en: {save_path}')

---
## ❓ Preguntas de reflexión

1. **¿Cuánto mejoró el fine-tuning respecto al baseline?**
2. **¿En qué época se activó el early stopping?** ¿Había signos de overfitting antes?
3. **¿El scheduler CosineAnnealingLR** ayudó a estabilizar el entrenamiento?
4. **¿Qué pasaría si descongeláramos más bloques** (n_blocks=6 o todo el backbone)?
5. **¿Vale la pena** intentar ResNet50 como comparativa para el informe?
